In [1]:
import pandas as pd

def make_lag_block_table(data: pd.DataFrame, T: int, d: int) -> pd.DataFrame:
    """
    参数
    ----
    data : DataFrame，含有 'index' 列为日期（如 20210104），其它列为股票代码
    T    : 与当前日的“间隔天数”
    d    : 连续取 d 天（生成 day_T+1 ... day_T+d 列）

    返回
    ----
    MultiIndex=(date, Code) 的表格，列为 day_T+1 ... day_T+d，且 date 为8位字符串（YYYYMMDD）
    """
    panel = data.copy()

    # 1) 把 index 列标准化为“日期”（内部用来排序/偏移），稍后再转回 8 位字符串
    try:
        panel["index"] = pd.to_datetime(panel["index"].astype(str), format="%Y%m%d")
    except Exception:
        panel["index"] = pd.to_datetime(panel["index"], errors="coerce")

    # 2) 宽转长
    long = (
        panel.melt(id_vars=["index"], var_name="Code", value_name="value")
          .rename(columns={"index": "date"})
          .sort_values(["Code", "date"])
    )

    # 3) 按股票分组生成 day_T+1 ... day_T+d
    g = long.groupby("Code", sort=False)["value"]
    for k in range(1, d + 1):
        offset = T + k
        col = f"day_T+{k}"
        if offset > 0:
            long[col] = g.shift(offset)
        elif offset == 0:
            long[col] = long["value"]
        else:
            long[col] = pd.NA

    # 4) 将 date 转回 8 位字符串，并返回以 (date, Code) 为 MultiIndex 的表格
    if not pd.api.types.is_datetime64_any_dtype(long["date"]):
        long["date"] = pd.to_datetime(long["date"], errors="coerce")
    long["date"] = long["date"].dt.strftime("%Y%m%d")

    out_cols = [f"day_T+{k}" for k in range(1, d + 1)]
    out = (
        long.set_index(["date", "Code"])[out_cols]
            .sort_index()
    )
    return out


In [2]:
import pandas as pd
import os

T = 5  # 间隔天数
d = 10 # 连续取 d 天
data1 = pd.read_feather("/home/user96/label_data1/can_trade_amt1.fea")
data1 = make_lag_block_table(data1, T=T, d=d)
data1 = data1.fillna(0)
num_cols = data1.select_dtypes(include="number").columns
means = data1.groupby(level="date")[num_cols].transform("mean")
stds  = data1.groupby(level="date")[num_cols].transform("std")  # ddof=1
stds = stds.replace(0, 1)
data1[num_cols] = (data1[num_cols] - means) / stds
data1 = data1.reset_index()
data1.to_feather(rf'/home/user96/score_file/hist.fea')
data1



,date,Code,day_T+1,day_T+2,day_T+3,day_T+4,day_T+5,day_T+6,day_T+7,day_T+8,day_T+9,day_T+10
0,20170103,000001,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,20170103,000002,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,20170103,000004,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,20170103,000005,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,20170103,000006,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...
10431962,20241231,688799,-0.399976,-0.352487,-0.377119,-0.370401,-0.365123,-0.426269,-0.448469,-0.399170,-0.411795,-0.584141
10431963,20241231,688800,0.454876,0.469732,0.096970,0.578087,4.142163,4.447264,0.252113,0.203153,-0.387102,-0.376196
10431964,20241231,688819,-0.392715,-0.352808,-0.410758,-0.337054,-0.368542,-0.410779,-0.347451,-0.391722,-0.371703,-0.327179
10431965,20241231,688981,6.099544,4.822398,5.733563,3.842510,2.083616,3.665575,2.079779,0.828162,5.097769,3.397900
